In [14]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline

analyzer = SentimentIntensityAnalyzer()

# VADER function to classify sentiment based on compound score
def get_vader_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05: return 'positive'
    elif score <= -0.05: return 'negative'
    else: return 'neutral'

# roBERTa training pipeline
classifier = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

Device set to use cpu


In [16]:
import pandas as pd

df = pd.read_csv("../test_sets/Sentiment-topic-test.tsv", sep="\t")

# apply VADER on testdata
df["vader_prediction"] = df["text"].apply(get_vader_sentiment)

# apply RoBERTa to test data
def run_roberta(text):
    result = classifier(str(text))[0]
    return result["label"].lower()

# add results to data
df["roberta_prediction"] = df["text"].apply(run_roberta)
df["disagreement"] = df["vader_prediction"] != df["roberta_prediction"]

# save the output
df.to_csv("sentiment_comparison_results.csv", index=False)

In [18]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score

file_path = "sentiment_comparison_results.csv"
df = pd.read_csv(file_path)

label_mapping = {"label_0": "negative", "label_1": "neutral", "label_2": "positive"}
df["roberta_prediction"] = df["roberta_prediction"].str.lower().map(label_mapping)

df["vader_correct"] = df["vader_prediction"] == df["sentiment"].str.lower()
df["roberta_correct"] = df["roberta_prediction"] == df["sentiment"].str.lower()
df["disagreement"] = df["vader_prediction"] != df["roberta_prediction"]

print(f"VADER Overall Accuracy: {accuracy_score(df['sentiment'], df['vader_prediction'])*100:.2f}%")
print(f"RoBERTa Overall Accuracy: {accuracy_score(df['sentiment'], df['roberta_prediction'])*100:.2f}%\n")

print("VADER")
print(classification_report(df['sentiment'], df['vader_prediction']))

print("\nROBERTA")
print(classification_report(df['sentiment'], df['roberta_prediction']))


print("Specific errors:")

vader_failures = df[(df['roberta_correct'] == True) & (df['vader_correct'] == False)].head(2)
for idx, row in vader_failures.iterrows():
    print(f"\n[VADER Failure] ID {row['sentence id']}: \"{row['text']}\"")
    print(f"Truth: {row['sentiment'].upper()} | VADER: {row['vader_prediction'].upper()} | RoBERTa: {row['roberta_prediction'].upper()}")

roberta_failures = df[(df['roberta_correct'] == False) & (df['vader_correct'] == True)].head(2)
for idx, row in roberta_failures.iterrows():
    print(f"\n[RoBERTa Failure] ID {row['sentence id']}: \"{row['text']}\"")
    print(f"Truth: {row['sentiment'].upper()} | VADER: {row['vader_prediction'].upper()} | RoBERTa: {row['roberta_prediction'].upper()}")

mutual_failures = df[(df['roberta_correct'] == False) & (df['vader_correct'] == False)].head(2)
for idx, row in mutual_failures.iterrows():
    print(f"\n[Hard Context] ID {row['sentence id']}: \"{row['text']}\"")
    print(f"Truth: {row['sentiment'].upper()} | VADER: {row['vader_prediction'].upper()} | RoBERTa: {row['roberta_prediction'].upper()}")

VADER Overall Accuracy: 60.00%
RoBERTa Overall Accuracy: 80.00%

VADER
              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
     neutral       1.00      0.33      0.50         3
    positive       0.50      1.00      0.67         4

    accuracy                           0.60        10
   macro avg       0.83      0.56      0.56        10
weighted avg       0.80      0.60      0.57        10


ROBERTA
              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
     neutral       0.75      1.00      0.86         3
    positive       0.80      1.00      0.89         4

    accuracy                           0.80        10
   macro avg       0.85      0.78      0.75        10
weighted avg       0.84      0.80      0.76        10

Specific errors:

[VADER Failure] ID 4: "The story of this movie is focused on Carl Brashear played by Cuba Gooding Jr. who wants to be the first African Amer